# 02: Econometric Model Estimation & Regression Analysis
**Repository:** `big5-player-valuation-econometrics`  
**Purpose:** Estimate Baseline, Positional, and Full OLS regressions to test Research Questions RQ1 (Attacking Premium) and RQ2 (Parabolic Age Trajectory).

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# Load cleaned dataset
df = pd.read_csv('data/processed/cleaned_players_data.csv')
df['log_market_value'] = np.log(df['market_value_eur'])
df['age_sq'] = df['age'] ** 2
print(f"Loaded {len(df)} player observations.")

## 2. Model Specifications
- **Model 1 (Baseline):** $$\ln(MV) = \beta_0 + \beta_1 \text{Age} + \beta_2 \text{Age}^2 + \beta_3 \text{Minutes} + \beta_4 \text{Contract} + \epsilon$$
- **Model 2 (Positional Dummies):** Adds position indicators (Reference: Defender)
- **Model 3 (Full Specification):** Adds position-specific performance metrics ($xG_{90}$, $ProgPasses_{90}$, $Tackles_{90}$)

In [ ]:
# Model 1: Baseline OLS
m1_formula = 'log_market_value ~ age + age_sq + minutes_played + contract_years'
model_1 = smf.ols(m1_formula, data=df).fit(cov_type='HC1')

# Model 2: Positional OLS
m2_formula = 'log_market_value ~ age + age_sq + minutes_played + contract_years + C(position, Treatment(reference="Defender"))'
model_2 = smf.ols(m2_formula, data=df).fit(cov_type='HC1')

# Model 3: Full OLS
m3_formula = 'log_market_value ~ age + age_sq + minutes_played + contract_years + C(position, Treatment(reference="Defender")) + xg_per90 + prog_passes_per90 + tackles_per90'
model_3 = smf.ols(m3_formula, data=df).fit(cov_type='HC1')

print("=== MODEL 3 SUMMARY (HC1 ROBUST SE) ===")
print(model_3.summary())

## 3. Regression Comparisons & Peak Age Calculation

In [ ]:
# Peak age calculation: -beta_age / (2 * beta_age_sq)
b_age = model_3.params['age']
b_age_sq = model_3.params['age_sq']
peak_age = -b_age / (2 * b_age_sq)
print(f"Estimated Valuation Peak Age: {peak_age:.2f} years")

# Model comparison summary
comparison = pd.DataFrame({
    'Model': ['Baseline', 'Positional', 'Full Specification'],
    'R_squared_adj': [model_1.rsquared_adj, model_2.rsquared_adj, model_3.rsquared_adj],
    'AIC': [model_1.aic, model_2.aic, model_3.aic],
    'BIC': [model_1.bic, model_2.bic, model_3.bic]
})
print(comparison)

## 4. Visualizing Curves & Exporting Figures (PNG + PDF)

In [ ]:
# Save Age Curve Plot
plt.figure(figsize=(8, 5))
sns.regplot(data=df, x='age', y='log_market_value', order=2, scatter_kws={'alpha': 0.25}, line_kws={'color': 'darkred', 'linewidth': 2})
plt.axvline(peak_age, color='black', linestyle='--', label=f'Peak Age ({peak_age:.1f} yrs)')
plt.title('Parabolic Age Valuation Curve')
plt.xlabel('Age')
plt.ylabel('ln(Market Value)')
plt.legend()
plt.savefig('figures/age_valuation_curves.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/age_valuation_curves.pdf', bbox_inches='tight')
plt.close()
print("Saved age curve figure in PNG and PDF.")